# 09 — IA: enriquecimiento de instrumentos
**Spec:** docs/specs/09_ia.md · **Requisitos:** R-28, R-29, R-30 · **Escribe:** `iol_ops.ai_runs`, `iol_ops.instrument_enrichment`, y reconstruye `iol_ops.instrument_mapping`

Completa lo que una regla no puede inferir: emisor, sector y ticker del subyacente, más los instrumentos que quedaron en `NEEDS_REVIEW`.
Solo se envían metadatos de instrumentos: ningún dato de clientes. La IA no pisa reglas, no decide calidad ni excluye filas (ADR-011).

In [0]:
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("run_evaluation", "true", ["true", "false"])

import json
import os
import sys
import uuid

sys.path.insert(0, os.path.abspath("../src"))
from iol_pipeline import ai, config, instruments, io

RUN_ID = dbutils.widgets.get("pipeline_run_id") or str(uuid.uuid4())
CATALOG = dbutils.widgets.get("catalog")
print("pipeline_run_id:", RUN_ID, "· modelo:", config.AI_MODEL, "· prompt:", config.AI_PROMPT_VERSION)

io.ensure_schemas(spark, catalog=CATALOG)
silver_table = io.table_name("silver", "trades", catalog=CATALOG)
mapping_table = io.table_name("ops", "instrument_mapping", catalog=CATALOG)
runs_table = io.table_name("ops", "ai_runs", catalog=CATALOG)
enrichment_table = io.table_name("ops", "instrument_enrichment", catalog=CATALOG)
quote_series_table = io.table_name("ops", "quote_series", catalog=CATALOG)  # la escribe el notebook 05

pipeline_run_id: 53c21210-4b46-4515-97b0-c9aa72b15d41 · modelo: databricks-qwen3-next-80b-a3b-instruct · prompt: instrument_enrichment_v1


## 1. Evaluación a ciegas de los modelos
Muestra etiquetada de `tests/fixtures/instrument_gold_labels.csv`. El modelo no recibe el tipo de las reglas: si no, lo copiaría y la accuracy saldría inflada.
Las respuestas quedan en caché: repetir la evaluación no vuelve a consultar.

In [0]:
if dbutils.widgets.get("run_evaluation") == "true":
    evaluation = ai.run_evaluation(spark, runs_table, config.AI_EVAL_MODELS, lambda model: ai.ai_query_caller(spark, model), RUN_ID)
    display(spark.createDataFrame(evaluation))

asset_type_accuracy,cedears,labeled,misses,model,ticker_accuracy,ticker_baseline_symbol,unusable
0.7833,22,60,SHELD: ACCION_LOCAL (esperado CEDEAR); SHELD: ticker None (esperado SHEL); YM34D: BONO_SUBSOBERANO (esperado ON); IRCPD: BONO_SUBSOBERANO (esperado ON); TLCTD: BONO_SUBSOBERANO (esperado ON); DNC7D: BONO_SUBSOBERANO (esperado ON); RUCDO: BONO_SUBSOBERANO (esperado ON); YMCJO: BONO_SUBSOBERANO (esperado ON); BACGO: BONO_SUBSOBERANO (esperado ON); MGCOD: BONO_SUBSOBERANO (esperado ON); BPY6D: BONO_SUBSOBERANO (esperado BONO_SOBERANO); TVPP: OTRO (esperado BONO_SOBERANO); TVPA: OTRO (esperado BONO_SOBERANO); BDC28: BONO_SOBERANO (esperado BONO_SUBSOBERANO),databricks-meta-llama-3-3-70b-instruct,0.9545,0.9091,0
0.8333,22,60,GOGLD: ticker GOGLD (esperado GOOGL); YM34D: BONO_SUBSOBERANO (esperado ON); IRCPD: OTRO (esperado ON); TLCTD: OTRO (esperado ON); DNC7D: OTRO (esperado ON); RUCDO: BONO_SUBSOBERANO (esperado ON); YMCJO: BONO_SUBSOBERANO (esperado ON); BACGO: BONO_SUBSOBERANO (esperado ON); MGCOD: OTRO (esperado ON); BDC28: BONO_SOBERANO (esperado BONO_SUBSOBERANO); D30A6: BONO_SOBERANO (esperado LETRA_LECAP_BONCAP),databricks-gpt-oss-120b,0.9545,0.9091,0
0.9,22,60,GOGLD: ticker GOOG (esperado GOOGL); SHELD: ACCION_LOCAL (esperado CEDEAR); BPY6D: BONO_SUBSOBERANO (esperado BONO_SOBERANO); TVPP: LETRA_LECAP_BONCAP (esperado BONO_SOBERANO); TVPA: LETRA_LECAP_BONCAP (esperado BONO_SOBERANO); BDC28: BONO_SOBERANO (esperado BONO_SUBSOBERANO); T30J6: BONO_SOBERANO (esperado LETRA_LECAP_BONCAP),databricks-qwen3-next-80b-a3b-instruct,0.9545,0.9091,0


## 2. Enriquecimiento

In [0]:
first = ai.run_enrichment(spark, mapping_table, runs_table, enrichment_table, ai.ai_query_caller(spark, config.AI_MODEL), RUN_ID)
print("1ra corrida:", first)
second = ai.run_enrichment(spark, mapping_table, runs_table, enrichment_table, ai.ai_query_caller(spark, config.AI_MODEL), RUN_ID)
print("2da corrida (tiene que consultar solo lo que falló):", second)
print("Mapeo:", instruments.run_instruments(spark, silver_table, mapping_table, enrichment_table=enrichment_table, quote_series_table=quote_series_table))

1ra corrida: {'instruments': 747, 'consulted': 0, 'statuses': {'VALID': 737, 'REJECTED': 10}, 'enriched_symbols': 1136, 'enrichment_rows_inserted': 0, 'enrichment_rows_updated': 0}
2da corrida (tiene que consultar solo lo que falló): {'instruments': 747, 'consulted': 0, 'statuses': {'VALID': 737, 'REJECTED': 10}, 'enriched_symbols': 1136, 'enrichment_rows_inserted': 0, 'enrichment_rows_updated': 0}
Mapeo: {'rows_inserted': 0, 'rows_updated': 0, 'symbols': 1445}


## 3. Resultado de la validación

In [0]:
runs = spark.table(runs_table)
display(ai.status_summary(runs))
display(ai.problem_examples(runs, config.AI_MODEL))

model,validation_status,count
databricks-meta-llama-3-3-70b-instruct,LOW_CONFIDENCE,8
databricks-meta-llama-3-3-70b-instruct,REJECTED,11
databricks-meta-llama-3-3-70b-instruct,VALID,728
databricks-qwen3-next-80b-a3b-instruct,REJECTED,11
databricks-qwen3-next-80b-a3b-instruct,VALID,743


symbol,validation_status,error
AAL,REJECTED,sector fuera del enum: 'TRANSPORTE'
ASR,REJECTED,sector fuera del enum: 'TRANSPORTE'
BDC28,REJECTED,asset_type fuera del enum: 'NEEDS_REVIEW'
CAAP,REJECTED,sector fuera del enum: 'TRANSPORTE'
CAR,REJECTED,sector fuera del enum: 'TRANSPORTE'
CCL,REJECTED,sector fuera del enum: 'TRANSPORTE'
DAL,REJECTED,sector fuera del enum: 'TRANSPORTE'
T13F6,REJECTED,sector fuera del enum: 'LETRA_LECAP_BONCAP'
T30J6,REJECTED,sector fuera del enum: 'LETRA_LECAP_BONCAP'
UAL,REJECTED,sector fuera del enum: 'TRANSPORTE'


## 4. Mapeo después de la IA

In [0]:
mapping = spark.table(mapping_table)
display(instruments.type_summary(mapping))
pending = instruments.needs_review(mapping)
print("Símbolos en NEEDS_REVIEW:", pending.count())
display(pending)
display(mapping.where("asset_type_source <> 'RULE'").select("symbol", "asset_type", "asset_type_source", "issuer", "description_canonical").orderBy("symbol"))
display(mapping.where("issuer IS NOT NULL").select("symbol", "asset_type", "asset_type_source", "issuer", "sector", "underlying_ticker_foreign").orderBy("symbol").limit(config.AI_EXAMPLE_ROWS))
print("Con emisor:", mapping.where("issuer IS NOT NULL").count(), "· con ticker extranjero:", mapping.where("underlying_ticker_foreign IS NOT NULL").count(), "· de", mapping.count())

asset_type,currency_code,asset_type_source,symbols
ACCION_LOCAL,ARS,AI,1
ACCION_LOCAL,ARS,RULE,65
ACCION_LOCAL,USD,RULE,22
BONO_SOBERANO,ARS,MANUAL,4
BONO_SOBERANO,ARS,RULE,53
BONO_SOBERANO,USD,RULE,41
BONO_SUBSOBERANO,ARS,MANUAL,1
BONO_SUBSOBERANO,ARS,RULE,10
BONO_SUBSOBERANO,USD,AI,1
BONO_SUBSOBERANO,USD,RULE,7


Símbolos en NEEDS_REVIEW: 0


symbol,currency_code,description_canonical,mapping_status


symbol,asset_type,asset_type_source,issuer,description_canonical
#MAV150560106,OTRO,MANUAL,null,pagare vto. 18/05/2026
BDC28,BONO_SUBSOBERANO,MANUAL,null,TITULOS DE DEUDA PUBLICA CLASE N° 23 A TASA VARIABLE CON VENCIMIENTO EN 2028
D30A6,LETRA_LECAP_BONCAP,AI,Nación Argentina,D30a6 Vto 30/04/2026
DGCE,ACCION_LOCAL,AI,Distribuidora De Gas Del Centro S.A.,Distribuidora De Gas Del Centro S.A.
SFD4D,BONO_SUBSOBERANO,AI,Gobierno Provincial de Santa Fe,TITULO DE DEUDA DE GOV. PROV SANTA FE 8.1% 11/12/34 USD MEP
SHELD,CEDEAR,MANUAL,Shell Plc,Shell Plc
TVPA,BONO_SOBERANO,MANUAL,República Argentina,Cupón vinculado al PBI US$ 2035 (Ley Arg)
TVPE,BONO_SOBERANO,MANUAL,República Argentina,Cupón vinculado al PBI Euros 2035
TVPP,BONO_SOBERANO,MANUAL,República Argentina,Cupón vinculado al PBI $ 2035 (Ley Arg)
TVPY,BONO_SOBERANO,MANUAL,República Argentina,Cupón vinculado al PBI US$ 2035 (Ley NY)


symbol,asset_type,asset_type_source,issuer,sector,underlying_ticker_foreign
A3,ACCION_LOCAL,RULE,Matba Rofex S.A.,FINANCIERO,null
AAP,CEDEAR,RULE,Advance Auto Parts Inc,CONSUMO,AAP
AAPL,CEDEAR,RULE,Apple Inc.,TECNOLOGIA,AAPL
AAPLC,CEDEAR,RULE,Apple Inc.,TECNOLOGIA,AAPL
AAPLD,CEDEAR,RULE,Apple Inc.,TECNOLOGIA,AAPL
ABBV,CEDEAR,RULE,Abbvie Inc.,SALUD,ABBV
ABBVD,CEDEAR,RULE,Abbvie Inc.,SALUD,ABBV
ABEV,CEDEAR,RULE,Ambev S.A.,CONSUMO,ABEV
ABEV3,CEDEAR,RULE,Ambev S.A.,CONSUMO,ABEV
ABEVD,CEDEAR,RULE,Ambev S.A.,CONSUMO,ABEV


Con emisor: 1134 · con ticker extranjero: 587 · de 1445


In [0]:
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "pipeline_run_id": RUN_ID, "rows_out": first["enriched_symbols"], "statuses": first["statuses"]}))